In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import os
import numpy as np
from scipy.stats import sem
import scipy
from cycler import cycler
# %matplotlib inline

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"
model_name = "CLIP_ViT_Vision" # CLIP_ViT_Vision | DeiT 
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
reps = [i for i in range(1,6)]

In [ ]:
import itertools
import json

dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]

all_combinations = {}

for r in range(1, 9):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

In [ ]:
import itertools
import json

dataset_name = ["INaturalist", "Cifar10", "Cifar100", "Food101", "DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]

all_combinations = {}

for r in range(1, 12):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

In [ ]:
scaling_coefficient = []
increment = 0.05
while increment < 1.05:
    scaling_coefficient.append(round(increment, 2))
    increment += 0.05

## Core Graphs

In [ ]:
for k in range(0, len(dataset_name)): # len(dataset_name)
    results_path = f"./{model_name}/Data/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    # 1-5
    results = {}

    for rep in reps:
        results[rep] = []
        path = f"{results_path}/{rep}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{rep}.json", f"Base_Linear_Probe_Results_{rep}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[rep].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[rep]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
        
        results[rep] = data

In [ ]:
mean_results = {}
top_error_bar = {}
bottom_error_bar = {}

for indice in indices:
    mean_results[indice] = []
    top_error_bar[indice] = []
    bottom_error_bar[indice] = []
    for file in range(len(results[1])):
        temp_acc = []
        for rep in range(1, len(results)+1):
            temp_acc.append(results[rep][file]["Classification_Accuracy"][indice])
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(temp_acc)-1, loc=np.mean(temp_acc), scale=sem(temp_acc))
        bottom_error_bar[indice].append(ci_low)
        top_error_bar[indice].append(ci_high)
        mean_results[indice].append(np.mean(temp_acc))

In [ ]:
fine_tune_mean = 0
fine_tune_top_error = 0
fine_tune_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    fine_tune_mean = np.mean(refined_acc)

    fine_tune_bottom_error, fine_tune_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
linear_probe_mean = 0
linear_probe_top_error = 0
linear_probe_bottom_error = 0
for k in range(6, 7):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    linear_probe_mean = np.mean(refined_acc)

    linear_probe_bottom_error, linear_probe_top_error = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=fine_tune_mean, scale=sem(refined_acc))

In [ ]:
train_size = []
for i in range(len(results[1])):
    train_size.append(results[1][i]["Train_Data_Size"][0][0])

my_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        "#528c4b", '#e377c2', '#7f7f7f', "#97b47c", '#17becf',
        '#a6cee3', '#b2df8a', '#fb9a99', "#c77b18", "#658a92"
]

plt.rcParams['axes.prop_cycle'] = cycler(color=my_colors)

plt.plot([0, train_size[-1]], [fine_tune_mean, fine_tune_mean], label="Fine Tuned", linestyle="--")
plt.plot([0, train_size[-1]], [linear_probe_mean, linear_probe_mean], label="Linear Probe", linestyle="--")
plt.plot([0, train_size[-1]], [1/397, 1/397], label="Base (Random)", linestyle='--')

for i in indices:
    plt.plot(train_size, mean_results[i], marker="o", label=f"Layer {i}")


plt.xlabel("Number of Training Images")
plt.ylabel("Classification Accuracy (%)")
plt.legend(
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)
plt.title(f"Task Matrices: CLIP ViT B/32 Vision - SUN397")
plt.savefig("./CLIP_ViT_Vision_SUN397", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Dataset groups
text_datasets = ['BLiMP', 'HANS', 'TREC-6']
vision_datasets = ['SUN397', 'GTSRB', 'MNIST']

# Data (aligned by dataset)
base =        [15.3, 63.4, 42.5, 65.3, 45.5, 48.9]
linear_probe =[38.1, 76, 75.1, 73.8, 86.8, 98.7]
task_matrix = [50, 82.3, 84.7, 74.8, 87.2, 99.03]
fine_tuned =  [60.5, 99.4, 93.2, 74.5, 98.7, 99.4]
baseline_random = [1/67 * 100, 1/2 * 100, 1/6 * 100, 1/397 * 100, 1/43 * 100, 1/10 * 100]

# Setup
fig, axs = plt.subplots(1, 2, figsize=(9, 7), sharey=True)

bar_width = 0.2

def plot_group(ax, indices, labels, remove_spine):
    x = np.arange(len(indices)) * 0.3

    colors = {
        'Base': 'skyblue',
        'Linear Probe': '#1f78b4',
        'Task Matrix': '#08306b',
        'Fine-Tuned': '#66c2a5'
    }

    for i, idx in enumerate(indices):
        # Values capped at 100
        vals = {
            'Base': min(base[idx], 100),
            'Linear Probe': min(linear_probe[idx], 100),
            'Task Matrix': min(task_matrix[idx], 100),
            'Fine-Tuned': min(fine_tuned[idx], 100)
        }

        # Draw bars in the specific order (back to front)
        order = ['Fine-Tuned', 'Task Matrix', 'Linear Probe', 'Base']

        for label in order:
            ax.bar(x[i], vals[label], width=bar_width, color=colors[label],
                   label=label if (i == 0) else "", alpha=1.0, zorder=order.index(label)+1)

        # Baseline line on top
        left = x[i] - bar_width / 2
        right = x[i] + bar_width / 2
        ax.hlines(y=baseline_random[idx], xmin=left, xmax=right, colors='red', linestyles='dashed',
                  label='Baseline (Random)' if i == 0 else "", linewidth=1.5, zorder=10)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=14)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    # Remove specific spines
    if remove_spine == 'right':
        ax.spines['right'].set_visible(False)
    elif remove_spine == 'left':
        ax.spines['left'].set_visible(False)

    for spine in ax.spines.values():
        spine.set_color('lightgrey')

    ax.tick_params(axis='both', which='both', length=0)

# Plot text datasets (left subplot)
plot_group(axs[0], [0, 1, 2], text_datasets, remove_spine='right')
axs[0].set_title("allMiniLM-L12-V2", fontsize=12)

# Plot vision datasets (right subplot)
plot_group(axs[1], [3, 4, 5], vision_datasets, remove_spine='left')
axs[1].set_title("CLIP ViT-B/32 Vision", fontsize=12)

# Shared Y label (grey)
fig.text(0.02, 0.5, 'Accuracy (%)', va='center', rotation='vertical', fontsize=13, color='grey')

# Legend (bottom)
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=5, frameon=True, fontsize=12)

plt.tight_layout(rect=[0.03, 0, 1, 1])
plt.savefig("./addp_results", dpi=600, bbox_inches='tight')
plt.show()

## Multi-Class Augmentation

### Single Graphs

In [ ]:
multi_class_scores = {}
multi_class_scores_error_top = {}
multi_class_scores_error_bottom = {}

for set in range(1,9):
    multi_class_scores[set] = {}
    multi_class_scores_error_top[set] = {}
    multi_class_scores_error_bottom[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores[set][matrix_name] = {}
        multi_class_scores_error_top[set][matrix_name] = {}
        multi_class_scores_error_bottom[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores[set][matrix_name][c] = {}
            multi_class_scores_error_top[set][matrix_name][c] = {}
            multi_class_scores_error_bottom[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
marker_list = []
x_acc = []
y_acc = []
labels = []
box_name = []
color_name = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels.append(label_name)
    box_name.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name.append(color_list[color_counter])
    color_counter += 1

    x_acc.append(multi_class_scores[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy[all_combinations[2][combo][0]])
    y_acc.append(multi_class_scores[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy[all_combinations[2][combo][1]])

In [ ]:
for x, y, label, marker, color_n in zip(x_acc, y_acc, labels, box_name, color_name):
    plt.scatter(x, y, label=label, marker=marker, color=color_n)

plt.axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
plt.axvline(x=1, color="grey", linestyle='--', linewidth=2)


plt.legend(
    loc='center left',
    bbox_to_anchor=(2.0, 0.5),
    ncol=2,
    fontsize='small'
)
plt.xlabel("Normalized Task 1 Accuracy")
plt.ylabel("Normalized Task 2 Accuracy")
plt.title("CLIP ViT Vision Multi-Task (2) Task Matrices")
plt.savefig("Multi_Class_Augmentation_2_Results.png", dpi=600, bbox_inches='tight')
plt.show()

### Double Graphs

In [ ]:
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"

multi_class_scores_1 = {}
multi_class_scores_error_top_1 = {}
multi_class_scores_error_bottom_1 = {}

for set in range(1,9):
    multi_class_scores_1[set] = {}
    multi_class_scores_error_top_1[set] = {}
    multi_class_scores_error_bottom_1[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores_1[set][matrix_name] = {}
        multi_class_scores_error_top_1[set][matrix_name] = {}
        multi_class_scores_error_bottom_1[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores_1[set][matrix_name][c] = {}
            multi_class_scores_error_top_1[set][matrix_name][c] = {}
            multi_class_scores_error_bottom_1[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores_1[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top_1[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom_1[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy_1 = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy_1[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
refining_type = "Increment_Training" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Reverse_Probe"

multi_class_scores_2 = {}
multi_class_scores_error_top_2 = {}
multi_class_scores_error_bottom_2 = {}

for set in range(1,9):
    multi_class_scores_2[set] = {}
    multi_class_scores_error_top_2[set] = {}
    multi_class_scores_error_bottom_2[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores_2[set][matrix_name] = {}
        multi_class_scores_error_top_2[set][matrix_name] = {}
        multi_class_scores_error_bottom_2[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores_2[set][matrix_name][c] = {}
            multi_class_scores_error_top_2[set][matrix_name][c] = {}
            multi_class_scores_error_bottom_2[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores_2[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top_2[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom_2[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy_2 = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy_2[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
x_acc_1 = []
y_acc_1 = []
labels_1 = []
box_name_1 = []
color_name_1 = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels_1.append(label_name)
    box_name_1.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name_1.append(color_list[color_counter])
    color_counter += 1

    x_acc_1.append(multi_class_scores_1[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy_1[all_combinations[2][combo][0]])
    y_acc_1.append(multi_class_scores_1[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy_1[all_combinations[2][combo][1]])

In [ ]:
x_acc_2 = []
y_acc_2 = []
labels_2 = []
box_name_2 = []
color_name_2 = []
marker_list = ["^", "H", "X", "D", "s", "*", "+", "p"]
color_list = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']
color_counter = 0
prev = None

for combo in range(len(all_combinations[2])):
    first = all_combinations[2][combo][0]
    matrix_name = ""
    for k in range(0, len(all_combinations[2][combo])):
        matrix_name += f"{all_combinations[2][combo][k]}_"
    label_name = f"{all_combinations[2][combo][0]}"
    for k in range(1, len(all_combinations[2][combo])):
        label_name += f", {all_combinations[2][combo][k]}"
    labels_2.append(label_name)
    box_name_2.append(marker_list[dataset_name.index(all_combinations[2][combo][0])])

    if first != prev:
        color_counter = 0
    
    prev = first
    color_name_2.append(color_list[color_counter])
    color_counter += 1

    x_acc_2.append(multi_class_scores_2[2][matrix_name][all_combinations[2][combo][0]][11] / fine_tuned_accuracy_2[all_combinations[2][combo][0]])
    y_acc_2.append(multi_class_scores_2[2][matrix_name][all_combinations[2][combo][1]][11] / fine_tuned_accuracy_2[all_combinations[2][combo][1]])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12,4), constrained_layout=True)

for x, y, label, marker, color_n in zip(x_acc_1, y_acc_1, labels_1, box_name_1, color_name_1):
    axes[0].scatter(x, y, label=label, marker=marker, color=color_n)

axes[0].axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
axes[0].axvline(x=1, color="grey", linestyle='--', linewidth=2)

for x, y, label, marker, color_n in zip(x_acc_2, y_acc_2, labels_2, box_name_2, color_name_2):
    axes[1].scatter(x, y, label=label, marker=marker, color=color_n)

axes[1].axhline(y=1, color="grey", linestyle='--', linewidth=2, label='Fine-Tuned Normalized Accuracy')
axes[1].axvline(x=1, color="grey", linestyle='--', linewidth=2)

axes[0].set_title("Full Train Multi-Task (2) Task Matrices")
axes[1].set_title("20% Train Multi-Task (2) Task Matrices")
fig.supylabel("Normalized Task 2 Accuracy")
fig.supxlabel("Normalized Task 1 Accuracy")

fig.legend(
    labels_1 + ["Fine-Tuned Normalized Accuracy"],
    ncol=5,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.02),
    bbox_transform=fig.transFigure,
    frameon=True
)

plt.savefig("Multi_Class_Augmentation_2_Results.pdf", dpi=600, bbox_inches="tight")
plt.show()

## All Train Images Layerwise Graphs

### Collecting Data

In [ ]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        indice = arr.index(max_val)
        num_img = data[indice]["Train_Data_Size"][i][0]
        best_acc[i] = (max_val, num_img)

    final = []
    num_img = []
    for i in indices:
        final.append(best_acc[i][0])
        num_img.append(best_acc[i][1])

    best_accuracy = max(final)
    index = final.index(best_accuracy)
    best_accuracy_num_images = num_img[index]

    return best_accuracy, best_accuracy_num_images, index

In [ ]:
def retrieve_val(path):
    best_data = pd.read_json(path)
    acc = {}
    for i in indices:
        acc[i] = best_data["Classification_Accuracy"][i]
    return acc

In [ ]:
# Task Matrix

best_layerwise_acc = {}
best_layerwise_img = {}
best_layerwise_index = {}

for ds_indice in range(len(dataset_name)):
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[ds_indice]}/{domain}"
    results = {}
    best_layerwise_acc[dataset_name[ds_indice]] = {}
    best_layerwise_img[dataset_name[ds_indice]] = {}
    best_layerwise_index[dataset_name[ds_indice]] = {}

    for rep in range(1,6):
        results[rep] = []
        path = f"{results_path}/{rep}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{rep}.json", f"Base_Linear_Probe_Results_{rep}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[rep].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = []
        for filepath in results[rep]:
            try:
                df = pd.read_json(filepath)
                data.append(df)
            except ValueError as ve:
                print(f"Failed to read JSON from file: {filepath} | Error: {ve}")
            except Exception as e:
                print(f"Other error with file {filepath}: {e}")
        results[rep] = data

    best_acc = []
    num_img = []
    index = []
    
    for rep in range(1,6):
        acc, img, ind = find_best_acc(results[rep])
        best_acc.append(acc)
        num_img.append(img)
        index.append(ind)
        best_layerwise_acc[dataset_name[ds_indice]][rep] = retrieve_val(f"{results_path}/{rep}/Entire_Transformation_Matrix_W/Standard_{img}_Results_{rep}.json")
        best_layerwise_img[dataset_name[ds_indice]][rep] = img
        best_layerwise_index[dataset_name[ds_indice]][rep] = index

In [ ]:
for ds_indice in range(len(dataset_name)):
    for rep in range(1,6):
        print(f"{dataset_name[ds_indice]} Repetition {rep}: {best_layerwise_acc[dataset_name[ds_indice]][rep]}")

In [ ]:
# Fine Tuned

fine_tuned_acc = {}

for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    fine_tuned_acc[dataset_name[k]] = refined_acc

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
# Linear Probe

linear_probe_acc = {}

for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Linear_Probe/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])
    linear_probe_acc[dataset_name[k]] = refined_acc
    
    print(f"{model_name} - {dataset_name[k]}: Linear Probe Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
# Ablation

best_layerwise_f_t_Classifier = {}
best_layerwise_f_t_Classifier_index = {}


def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)

    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        best_acc[i] = max_val

    final = []
    for i in indices:
        final.append(best_acc[i])

    best_accuracy = max(final)
    index = final.index(best_accuracy)

    return best_accuracy, index

# Ablation Base with Fine Tuned Classifier Head
# big_data = {i: {} for i in range(len(dataset_name))}

for k in range(len(dataset_name)):
    best_layerwise_f_t_Classifier[dataset_name[k]] = {}
    best_layerwise_f_t_Classifier_index[dataset_name[k]] = {}
    said=k
    results_path = f"./{model_name}/Data/Core/{refining_type}/{refiner}/{dataset_name[k]}/{domain}" # /Entire_Transformation_Matrix_W"

    big_data = {}
    for i in range(1,6):
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [f"Base_Fine_Tuned_Classifier_Results_{i}.json"]: 
                    file_path = os.path.join(path, filename)
                    if os.path.isfile(file_path):
                        results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = [pd.read_json(i) for i in results[i]]
        results[i] = data
        big_data[i] = results[i]
    
    best_acc = []
    index = []
    print(f"{model_name} - {dataset_name[said]}: Ablation Base with Fine-Tuned Classifier")
    for i in range(1,6):
        acc, ind = find_best_acc(big_data[i])
        best_acc.append(acc)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Transformation Layer (0-11): {index[i-1]}")
        best_layerwise_f_t_Classifier[dataset_name[k]][i] = retrieve_val(f"{results_path}/{i}/Entire_Transformation_Matrix_W/Base_Fine_Tuned_Classifier_Results_{i}.json")
        best_layerwise_f_t_Classifier_index[dataset_name[k]][i] = ind
    
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")

In [ ]:
for ds_indice in range(len(dataset_name)):
    for rep in range(1,6):
        print(best_layerwise_f_t_Classifier[dataset_name[ds_indice]][rep])

### Graphing Data

In [ ]:
# best_layerwise_acc = {}
# best_layerwise_img = {}
# best_layerwise_index = {}
# fine_tuned_acc = {}
# linear_probe_acc = {}
# best_layerwise_f_t_Classifier = {}
# best_layerwise_f_t_Classifier_index = {}

In [ ]:
task_matrix_indice_mean_acc = {}
ablation_indice_mean_acc = {}

In [ ]:
for ds_indice in range(len(dataset_name)):
    task_matrix_indice_mean_acc[dataset_name[ds_indice]] = []
    ablation_indice_mean_acc[dataset_name[ds_indice]] = []
    for idx in indices:
        temp_acc = []
        temp_ablation_acc = []
        for rep in range(1,6):
            temp_acc.append(best_layerwise_acc[dataset_name[ds_indice]][rep][idx])
            temp_ablation_acc.append(best_layerwise_f_t_Classifier[dataset_name[ds_indice]][rep][idx])
        task_matrix_indice_mean_acc[dataset_name[ds_indice]].append(np.mean(temp_acc))
        ablation_indice_mean_acc[dataset_name[ds_indice]].append(np.mean(temp_ablation_acc))

In [ ]:
for ds_indice in range(len(dataset_name)):
    # print(task_matrix_indice_mean_acc[dataset_name[ds_indice]])
    print(ablation_indice_mean_acc[dataset_name[ds_indice]])

In [ ]:
# Consolidated
layerwise_results_path = f"./{model_name}/Graphs/{refining_type}/{refiner}"
os.makedirs(layerwise_results_path, exist_ok=True)

In [ ]:
import numpy as np

# ==============================================================================
# LAYER-BY-LAYER ACCURACIES FOR ALL DATASETS IN PYTHON LIST FORMAT
# ==============================================================================

# Dataset 1: Emotion Classification Data
emotion_data = {
    'title': 'Emotion',
    'task_matrix': [
        0.6236, 0.6594, 0.6486, 0.6342, 0.6256, 0.6332, 0.6300, 0.6348,
        0.6166, 0.6310, 0.6490, 0.6460, 0.6412, 0.6326, 0.6176, 0.6128,
        0.6248, 0.6212, 0.5930, 0.5754, 0.5996, 0.5904, 0.5756, 0.5956
    ],
    'base_ft_cls': [
        0.2350, 0.2350, 0.2350, 0.2350, 0.2295, 0.2350, 0.2350, 0.2075,
        0.3210, 0.2350, 0.2450, 0.3350, 0.3350, 0.2350, 0.2085, 0.0590,
        0.0590, 0.0590, 0.0715, 0.0750, 0.0625, 0.2050, 0.2085, 0.2085
    ],
    'fine_tuned_baseline': 0.914,
    'linear_probe_baseline': 0.589
}

# Dataset 2: Banking77 Classification Data
banking77_data = {
    'title': 'Banking77',
    'task_matrix': [
        0.8072, 0.8302, 0.8261, 0.8194, 0.8200, 0.8122, 0.8040, 0.8119,
        0.7861, 0.7950, 0.7891, 0.7977, 0.7881, 0.7820, 0.7583, 0.7487,
        0.7853, 0.7931, 0.7954, 0.7900, 0.7916, 0.7706, 0.7780, 0.8273
    ],
    'base_ft_cls': [
        0.0125, 0.0127, 0.0135, 0.0145, 0.0135, 0.0135, 0.0135, 0.0129,
        0.0133, 0.0153, 0.0147, 0.0147, 0.0141, 0.0141, 0.0113, 0.0127,
        0.0120, 0.0113, 0.0115, 0.0113, 0.0107, 0.0113, 0.0121, 0.0137
    ],
    'fine_tuned_baseline': 0.920,
    'linear_probe_baseline': 0.643
}

# Dataset 3: TREC-6 Classification Data
trec6_data = {
    'title': 'TREC-6',
    'task_matrix': [
        0.7620, 0.7880, 0.7868, 0.7912, 0.8138, 0.8280, 0.8262, 0.8248,
        0.8196, 0.8318, 0.8410, 0.8382, 0.8326, 0.8308, 0.8322, 0.8258,
        0.8144, 0.8210, 0.7908, 0.8094, 0.8238, 0.8166, 0.7690, 0.8406
    ],
    'base_ft_cls': [
        0.1992, 0.1990, 0.1990, 0.1990, 0.1910, 0.1976, 0.1976, 0.1990,
        0.1982, 0.1998, 0.2020, 0.2038, 0.2062, 0.2068, 0.2096, 0.1992,
        0.1970, 0.1982, 0.1886, 0.1960, 0.1974, 0.1988, 0.2100, 0.2134
    ],
    'fine_tuned_baseline': 0.951,
    'linear_probe_baseline': 0.798
}

# Dataset 4: ATIS Intent Classification Data
atis_data = {
    'title': 'ATIS',
    'task_matrix': [
        0.9296, 0.9412, 0.9398, 0.9420, 0.9496, 0.9454, 0.9508, 0.9468,
        0.9398, 0.9388, 0.9364, 0.9364, 0.9394, 0.9374, 0.9338, 0.9278,
        0.9324, 0.9302, 0.9222, 0.9350, 0.9334, 0.9282, 0.9188, 0.9378
    ],
    'base_ft_cls': [
        0.0062, 0.1602, 0.1602, 0.1584, 0.1604, 0.1418, 0.0900, 0.0064,
        0.0102, 0.0208, 0.0206, 0.0108, 0.0152, 0.0062, 0.0188, 0.0202,
        0.0178, 0.0174, 0.0152, 0.0162, 0.0212, 0.0210, 0.0240, 0.0212
    ],
    'fine_tuned_baseline': 0.9782,
    'linear_probe_baseline': 0.8932
}

# Dataset 5: SNLI Classification Data
snli_data = {
    'title': 'SNLI',
    'task_matrix': [
        0.525, 0.563, 0.566, 0.570, 0.583, 0.596, 0.601, 0.608,
        0.618, 0.634, 0.646, 0.667, 0.699, 0.724, 0.744, 0.758,
        0.761, 0.761, 0.758, 0.754, 0.749, 0.737, 0.731, 0.728
    ],
    'base_ft_cls': [
        0.333, 0.333, 0.333, 0.333, 0.333, 0.333, 0.333, 0.333,
        0.333, 0.333, 0.333, 0.333, 0.333, 0.333, 0.335, 0.333,
        0.334, 0.334, 0.334, 0.334, 0.334, 0.334, 0.337, 0.334
    ],
    'fine_tuned_baseline': 0.891,
    'linear_probe_baseline': 0.676
}

# Dataset 6: BLiMP Grammaticality Judgements
blimp_data = {
    'title': 'BLiMP',
    'task_matrix': [
        0.6593, 0.6788, 0.7145, 0.7130, 0.7488, 0.7528, 0.7513, 0.7423,
        0.7303, 0.7278, 0.7298, 0.7360, 0.7348, 0.7315, 0.7183, 0.7145,
        0.7223, 0.7153, 0.7073, 0.7173, 0.7243, 0.7210, 0.7155, 0.7015
    ],
    'base_ft_cls': [
        0.0145, 0.0155, 0.0148, 0.0148, 0.0145, 0.0143, 0.0148, 0.0195,
        0.0200, 0.0153, 0.0128, 0.0128, 0.0138, 0.0138, 0.0143, 0.0203,
        0.0155, 0.0155, 0.0135, 0.0125, 0.0148, 0.0180, 0.0180, 0.0153
    ],
    'fine_tuned_baseline': 0.830,
    'linear_probe_baseline': 0.5865
}

# A list containing all dataset dictionaries for easy access
all_datasets_data = [
    emotion_data,
    banking77_data,
    trec6_data,
    atis_data,
    snli_data,
    blimp_data
]

# You can now access the data like this:
# print("Emotion Task Matrix data for layer 1:", all_datasets_data[0]['task_matrix'][0])
# print("BLiMP Fine-Tuned baseline:", all_datasets_data[5]['fine_tuned_baseline'])

### Vision Layerwise Graph

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(24,13), constrained_layout=True)

row_num = 0
col_num = 0

for ds_indice in range(len(dataset_name)):
    axes[row_num, col_num].plot(indices, task_matrix_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Task Matrix")
    axes[row_num, col_num].plot(indices, ablation_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Base w/Fine-Tuned Classifier")
    axes[row_num, col_num].plot(indices, [np.mean(fine_tuned_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Fine-Tuned")
    axes[row_num, col_num].plot(indices, [np.mean(linear_probe_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Linear Probe")
    axes[row_num, col_num].set_title(dataset_name[ds_indice], fontweight="bold")
    col_num += 1
    if col_num == 4:
        col_num = 0
        row_num += 1

for ax in axes[0, :]:
    ax.tick_params(axis='x', labelbottom=False)

fig.supylabel('Classification Accuracy')
fig.supxlabel('Layer')
labels = ["Task Matrix", "Base w/Fine-Tuned Classifier", "Fine-Tuned", "Linear Probe"]
# fig.suptitle("CLIP ViT Vision Best Layerwise Accuracy", fontsize=25)

fig.legend(
    labels,
    ncol=4,
    loc = "upper center",
    # fontsize=21,
    # markerscale=2,
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
)

# plt.savefig("Best_Layerwise_Acc_CLIP_ViT_Vision.pdf", dpi=600, bbox_inches='tight')
plt.show()

### Individual Vision Graphs

In [ ]:
# Individual Graphs

for ds_indice in range(len(dataset_name)):
    plt.plot(indices, task_matrix_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Task Matrix")
    plt.plot(indices, ablation_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Base w/Fine-Tuned Classifier")
    plt.plot(indices, [np.mean(fine_tuned_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Fine-Tuned")
    plt.plot(indices, [np.mean(linear_probe_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Linear Probe")
    plt.legend(
        loc="center left",
        bbox_to_anchor=(1.0, 0.5)
    )
    plt.title(f"CLIP ViT Vision {dataset_name[ds_indice]} Best Layerwise Results")
    plt.xlabel("Layer")
    plt.ylabel("Classification Accuracy")
    plt.savefig(f"{layerwise_results_path}/{dataset_name[ds_indice]}_Layerwise_Accuracy", dpi=600, bbox_inches='tight')
    plt.show()

### Conslidated Layerwise Graphs

In [ ]:
# Consolidated Layerwise Graphs

import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'xtick.labelsize': 15,  # X-axis tick label size
    'ytick.labelsize': 15,  # Y-axis tick label size
    'xtick.major.size': 10,  # X-axis major tick size
    'ytick.major.size': 10,  # Y-axis major tick size
    'xtick.major.width': 2,  # X-axis major tick width
    'ytick.major.width': 2,  # Y-axis major tick width
})

fig = plt.figure(figsize=(36,20), constrained_layout=True)
gs = gridspec.GridSpec(4, 4, figure=fig, wspace=0.05, hspace=0.1) # hspace=0.6

row_num = 0
col_num = 0

# CLIP VIT Vision
for ds_indice in range(len(dataset_name)):
    axes = fig.add_subplot(gs[row_num, col_num])
    axes.plot(indices, task_matrix_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Task Matrix")
    axes.plot(indices, ablation_indice_mean_acc[dataset_name[ds_indice]], marker = "o", label="Base w/Fine-Tuned Classifier")
    axes.plot(indices, [np.mean(fine_tuned_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Fine-Tuned")
    axes.plot(indices, [np.mean(linear_probe_acc[dataset_name[ds_indice]])]*12, linestyle="--", label="Linear Probe")
    axes.set_title(dataset_name[ds_indice], fontweight="bold", fontsize=16)
    col_num += 1
    if col_num == 4:
        col_num = 0
        row_num += 1

col_num = 0
# RoBERTa 
text_dataset_names = ["Emotion", "Banking77", "TREC-6", "ATIS", "SNLI", "BLiMP"]
text_indices = [i for i in range(24)]
for ds_indice in range(len(text_dataset_names)):
    axes = fig.add_subplot(gs[row_num, col_num])
    axes.plot(text_indices, all_datasets_data[ds_indice]['task_matrix'], marker = "o", label="Task Matrix")
    axes.plot(text_indices, all_datasets_data[ds_indice]['base_ft_cls'], marker = "o", label="Base w/Fine-Tuned Classifier")
    axes.plot(text_indices, [all_datasets_data[ds_indice]['fine_tuned_baseline']]*24, linestyle="--", label="Fine-Tuned")
    axes.plot(text_indices, [all_datasets_data[ds_indice]['linear_probe_baseline']]*24, linestyle="--", label="Linear Probe")
    axes.set_title(text_dataset_names[ds_indice], fontweight="bold", fontsize=16)
    col_num += 1
    if col_num == 4:
        col_num = 0
        row_num += 1

axes = fig.add_subplot(gs[3,3])
axes.axis("off")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], label='Task Matrix', color="tab:blue", marker="o", markersize=12, linewidth=4),
    Line2D([0], [0], label='Base w/Fine-Tuned Classifier', color="tab:orange", marker="o", markersize=12, linewidth=4),
    Line2D([0], [0], label='Fine-Tuned', color="tab:green", linestyle="--", linewidth=4),
    Line2D([0], [0], label="Linear Probe", color="tab:red", linestyle="--", linewidth=4)
]
axes.legend(handles=legend_elements, loc='center', frameon=True, fontsize=30)

fig.supylabel('Classification Accuracy', fontsize=28, x=-0.02)
fig.supxlabel('Layer', fontsize=28, x=0.5033)

fig.text(0.505, 0.51, "RoBERTa", ha='center', va='center', fontsize=26)
fig.suptitle("CLIP ViT Vision Tower", x=0.505, y=1.02, fontsize=26)

plt.savefig("Model_Consolidated_Layerwise_Graph.pdf", dpi=600, bbox_inches='tight')
plt.show()

## Scaling Coefficient

In [ ]:
scaling_num = {}

results_path = f"./{model_name}/Data/Scaling_Coefficient/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
print(os.path.isdir(results_path))

for ds_indice in range(1, 3):
    scaling_num[ds_indice] = {}
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        temp_acc = {}
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
            temp_acc[name] = {i: {j: [] for j in scaling_coefficient} for i in indices}
        
        scaling_num[ds_indice][matrix_name] = {}
        
        for rep in range(1,6):
            try: 
                file_path = f"{results_path}/{ds_indice}/{domain}/{matrix_name}/{rep}.json"
                with open(file_path, "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")

            for c in all_combinations[ds_indice][combo]:
                for i in indices:
                    for j in scaling_coefficient:
                        temp_acc[c][i][j].append(data["Classification_Accuracy"][c][str(j)][str(i)])
            
        for c in all_combinations[ds_indice][combo]:
            scaling_num[ds_indice][matrix_name][c] = {}
            for j in scaling_coefficient:
                scaling_num[ds_indice][matrix_name][c][j] = {}
                for i in indices:
                    scaling_num[ds_indice][matrix_name][c][j][i] = np.mean(temp_acc[c][i][j])

In [ ]:
y_acc = {}
for ds_indice in range(1,2):
    y_acc[ds_indice] = {}
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
        y_acc[ds_indice][matrix_name] = {}
        for c in all_combinations[ds_indice][combo]:
            y_acc[ds_indice][matrix_name][c] = {}
            for i in indices:
                y_acc[ds_indice][matrix_name][c][i] = []
                for scale in scaling_coefficient:
                    y_acc[ds_indice][matrix_name][c][i].append(scaling_num[ds_indice][matrix_name][c][scale][i])

In [ ]:
my_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        "#528c4b", '#e377c2', '#7f7f7f', "#97b47c", '#17becf',
        '#a6cee3', '#b2df8a', '#fb9a99', "#c77b18", "#658a92"
]

plt.rcParams['axes.prop_cycle'] = cycler(color=my_colors)

for ds_indice in range(1,2):
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
        for c in all_combinations[ds_indice][combo]:
            for i in indices:
                plt.plot(scaling_coefficient, y_acc[ds_indice][matrix_name][c][i], label=f"Layer {i}", marker="o")
            plt.xlabel("Scaling Coefficient")
            plt.ylabel("Classification Accuracy")
            plt.title(f"{matrix_name}")
            plt.legend(
                loc="center left",
                bbox_to_anchor=(1.0, 0.5)
            )
            plt.show()

In [ ]:
for ds_indice in range(1,2):
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
        for c in all_combinations[ds_indice][combo]:
            for i in indices:
                max_val = max(y_acc[ds_indice][matrix_name][c][i])
                coef = y_acc[ds_indice][matrix_name][c][i].index(max_val)
                print(f"{matrix_name} - {c} Layer {i}: {max_val} Coefficient {scaling_coefficient[coef]}")

In [ ]:
y_acc = {}
for ds_indice in range(2,3):
    y_acc[ds_indice] = {}
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
        y_acc[ds_indice][matrix_name] = {}
        for c in all_combinations[ds_indice][combo]:
            y_acc[ds_indice][matrix_name][c] = {}
            for i in indices:
                y_acc[ds_indice][matrix_name][c][i] = []
                for scale in scaling_coefficient:
                    y_acc[ds_indice][matrix_name][c][i].append(scaling_num[ds_indice][matrix_name][c][scale][i])

In [ ]:
for ds_indice in range(2,3):
    for combo in range(len(all_combinations[ds_indice])):
        matrix_name = ""
        for name in all_combinations[ds_indice][combo]:
            matrix_name += f"{name}_"
        for c in all_combinations[ds_indice][combo]:
            for i in indices:
                max_val = max(y_acc[ds_indice][matrix_name][c][i])
                coef = y_acc[ds_indice][matrix_name][c][i].index(max_val)
                print(f"{matrix_name} - {c} Layer {i}: {max_val} Coefficient {scaling_coefficient[coef]}")

## All Multi-Class Numbers

In [ ]:
multi_class_scores = {}
multi_class_scores_error_top = {}
multi_class_scores_error_bottom = {}

for set in range(1,9):
    multi_class_scores[set] = {}
    multi_class_scores_error_top[set] = {}
    multi_class_scores_error_bottom[set] = {}
    for combo in range(len(all_combinations[set])):
        results_path = f"./{model_name}/Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Performance"
        temp_acc = {}
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
            temp_acc[c] = {i: [] for i in indices}
        
        for rep in range(1,6):
            try: 
                with open(f"{results_path}/{set}/{domain}/{matrix_name}/{rep}.json", "r") as file:
                    data = json.load(file)
            except FileNotFoundError:
                print("Error: data.json not found.")
            for c in all_combinations[set][combo]:
                for i in indices:
                    temp_acc[c][i].append(data["Classification_Accuracy"][c][str(i)])
        # temp_acc (dataset_name, layer, accuracy)

        multi_class_scores[set][matrix_name] = {}
        multi_class_scores_error_top[set][matrix_name] = {}
        multi_class_scores_error_bottom[set][matrix_name] = {}

        for c in all_combinations[set][combo]:
            multi_class_scores[set][matrix_name][c] = {}
            multi_class_scores_error_top[set][matrix_name][c] = {}
            multi_class_scores_error_bottom[set][matrix_name][c] = {}
            for i in indices:
                fine_tune_mean = np.mean(temp_acc[c][i])
                multi_class_scores[set][matrix_name][c][i] = fine_tune_mean
                b, t = scipy.stats.t.interval(0.95, df=len(temp_acc[c][i])-1, loc=fine_tune_mean, scale=sem(temp_acc[c][i]))
                multi_class_scores_error_top[set][matrix_name][c][i] = b
                multi_class_scores_error_bottom[set][matrix_name][c][i] = t

In [ ]:
fine_tuned_accuracy = {}
for k in range(0, len(dataset_name)):
    refined_acc = []
    df = pd.read_json(f"./{model_name}/Data/Core/{refining_type}/Refined_Accuracy/Fine_Tuned/{dataset_name[k]}_Accuracy.json")

    for i in range(5):
        refined_acc.append(df["Accuracy"][i])

    print(f"{model_name} - {dataset_name[k]}: Fine Tuned Accuracy")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(refined_acc)-1, loc=np.mean(refined_acc), scale=sem(refined_acc))
    print(f"Fine-Tuned Average: {np.mean(refined_acc)}, Error: +- {ci_high-np.mean(refined_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}\n")
    fine_tuned_accuracy[dataset_name[k]] = np.mean(refined_acc)

In [ ]:
normalized_acc = {}

for set in range(1,9):
    normalized_acc[set] = {}
    for combo in range(len(all_combinations[set])):
        matrix_name = ""
        for c in all_combinations[set][combo]:
            matrix_name += f"{c}_"
        normalized_acc[set][matrix_name] = {}

        for ds_name in all_combinations[set][combo]:
            normalized_acc[set][matrix_name][ds_name] = []
            for layer in range(12):
                normalized_acc[set][matrix_name][ds_name].append(multi_class_scores[set][matrix_name][ds_name][layer] / fine_tuned_accuracy[ds_name])

In [ ]:
combined_acc = {}

for layer in range(12):
    combined_acc[layer] = {}
    for set in range(1,9):
        combined_acc[layer][set] = []
        for combo in range(len(all_combinations[set])):
            matrix_name = ""
            for c in all_combinations[set][combo]:
                matrix_name += f"{c}_"
            
            for ds_name in all_combinations[set][combo]:
                combined_acc[layer][set].append(normalized_acc[set][matrix_name][ds_name][layer])

In [ ]:
mean_combined_acc = {}

for layer in range(12):
    mean_combined_acc[layer] = []
    for set in range(1,9):
        mean_combined_acc[layer].append(np.mean(combined_acc[layer][set]))

In [ ]:
setter = [i for i in range(1,9)]

### Multiple Plots

In [ ]:
for layer in range(12):
    for set in range(1,9):
        for val in combined_acc[layer][set]:
            plt.scatter(set, val, color="lightblue")
    plt.plot(setter, mean_combined_acc[layer], color="orange")
    plt.xlabel("Number of Datasets")
    plt.ylabel("Normalized Classification Accuracy")
    plt.title(f"Layer {layer}")
    plt.show()

### Single Plot

In [ ]:
from matplotlib.lines import Line2D

plt.rcParams.update({
    'xtick.labelsize': 20,  # X-axis tick label size
    'ytick.labelsize': 20,  # Y-axis tick label size
    'xtick.major.size': 12,  # X-axis major tick size
    'ytick.major.size': 12,  # Y-axis major tick size
    'xtick.major.width': 3,  # X-axis major tick width
    'ytick.major.width': 3,  # Y-axis major tick width
})

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(52, 26), constrained_layout=True)

row_num = 0
col_num = 0

for layer in range(12):
    for set in range(1,9):
        for val in combined_acc[layer][set]:
            axes[row_num, col_num].scatter(set, val, color="lightblue", label="Task Matrix")
    axes[row_num, col_num].plot(setter, mean_combined_acc[layer], color="orange", label="Average Task Matrix")
    axes[row_num, col_num].set_title(f"Layer {layer}", fontweight="bold", fontsize=24)
    col_num += 1
    if col_num == 4:
        col_num = 0
        row_num += 1    

fig.supylabel('Normalized Classification Accuracy', fontsize=26)
fig.supxlabel('Number of Datasets', fontsize=26)

task_matrix_handle = Line2D([0], [0], marker='o', color='lightblue', label="Task Matrix", markersize=10)
avg_task_matrix_handle = Line2D([0], [0], color='orange', label="Average Task Matrix", linewidth=3)

# Add the custom legend
fig.legend(
    handles=[task_matrix_handle, avg_task_matrix_handle],
    labels=["Task Matrix", "Average Task Matrix"],
    ncol=2,
    loc="upper center",
    fontsize=26,
    markerscale=2,
    bbox_to_anchor=(0.5, 0),
    bbox_transform=fig.transFigure,
)

In [ ]:
plt.savefig(f"Multi_Class_Acc_{model_name}_{refining_type}.pdf", dpi=600, bbox_inches='tight')
plt.show()